In [ ]:
import os
import re
from datetime import datetime
import pandas as pd
from bs4 import BeautifulSoup
from typing import Iterable
from helpers import get_request

In [2]:
filename = "./article_links_16-09-25_12-21-53_with_contents"
articles_df = pd.read_csv(f"{filename}.csv")
articles_df

,link,type,time_ago,headline,content
0,/news/scorecard-noche-ufc-san-antonio,Fight Coverage,20 hours ago,The Scorecard | Noche UFC,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca..."
1,/news/kyle-daukaus-darce-knight-returns,Athletes,21 hours ago,KYLE DAUKAUS | THE D’ARCE KNIGHT RETURNS,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca..."
2,/news/live-results-official-scorecards-match-r...,Results,2 days ago,Results + Scorecards | Canelo vs Crawford,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca..."
3,/news/main-card-live-results-fight-recaps-high...,Results,2 days ago,Main Card Results | Noche UFC,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca..."
4,/news/prelim-live-results-fight-recaps-highlig...,Results,2 days ago,Prelim Results | Noche UFC,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca..."
...,...,...,...,...,...
724,/news/umar-nurmagomedov-is-ready-to-go-into-de...,Athletes,8 months ago,Umar Nurmagomedov Is Ready To Go Into Deep Waters,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca..."
725,/news/grant-dawson-very-good-year-ufc-311,Athletes,8 months ago,Grant Dawson's Very Good Year,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca..."
726,/news/islam-makhachev-wants-to-eliminate-all-d...,Athletes,8 months ago,Islam Makhachev Wants To Eliminate All Doubt,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca..."
727,/news/prelim-results-highlights-winner-intervi...,Results,8 months ago,Prelim Results | UFC Fight Night: Dern vs Ribas 2,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca..."


In [35]:
author_pattern = r"By ([A-Za-z\s.]+).*"
social_media_pattern = r"@([A-Za-z]+)"

def get_page_datas(page_contents: Iterable[str]):
    soups = [BeautifulSoup(content, "html.parser") for content in page_contents]

    def get_titles():
        for soup in soups:
            title_element = soup.find("meta", property="og:title")
            yield title_element.get("content") if title_element else None
    
    def get_descriptions():
        for soup in soups:
            description_element = soup.find("meta", property="og:description")
            yield description_element.get("content") if description_element else None
    
    def get_published_times():
        for soup in soups:
            published_time_element = soup.find("meta", property="article:published_time")
            yield datetime.fromisoformat(published_time_element.get("content"))\
                if published_time_element\
                else None
    
    def get_modified_times():
        for soup in soups:
            modified_time_element = soup.find("meta", property="article:modified_time")
            yield datetime.fromisoformat(modified_time_element.get("content"))\
                if modified_time_element\
                else None

    def get_authors():
        for soup in soups:
            credit_element = soup.find("div", "c-hero__article-credit")

            if credit_element:
                credit = credit_element.text.strip().replace("\n", "")
                author_match = re.search(author_pattern, credit)
                yield author_match.group(1).strip() if author_match else None
            else:
                yield None

    def get_social_medias():
        for soup in soups:
            credit_element = soup.find("div", "c-hero__article-credit")

            if credit_element:
                credit = credit_element.text.strip().replace("\n", "")
                social_media_match = re.search(social_media_pattern, credit)
                yield social_media_match.group(1).strip() if social_media_match else None
            else:
                yield None
    
    def get_texts():
        for soup in soups:
            container = soup.find("div", "l-two-col__content")
            yield container.text.strip() if container else None
    
    def get_image_srcs():
        for soup in soups:
            images_srcs: list[str] = [image.get("src") for image in soup.find_all("img")]
            yield [
                src if src.startswith("https") else f"https://ufc.com{src}"
                for src in images_srcs if src
            ]
    
    return {
        "title": get_titles(),
        "description": get_descriptions(),
        "published_time": get_published_times(),
        "modified_time": get_modified_times(),
        "author": get_authors(),
        "social_media": get_social_medias(),
        "text": get_texts(),
        "images_srcs": get_image_srcs(),
    }

In [36]:
datas = get_page_datas(articles_df["content"])
for key in datas.keys():
    articles_df[key] = list(datas[key])
articles_df

,link,type,time_ago,headline,content,title,description,published_time,modified_time,author,social_media,text,images_srcs
0,/news/scorecard-noche-ufc-san-antonio,Fight Coverage,20 hours ago,The Scorecard | Noche UFC,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca...",The Scorecard | Noche UFC,A Look Back At The Biggest Winners From Frost ...,2025-09-15 15:48:46-05:00,2025-09-15 16:15:16-05:00,Steve Latrell,None,The third annual Mexican Independence celebrat...,[https://ufc.com/images/styles/background_imag...
1,/news/kyle-daukaus-darce-knight-returns,Athletes,21 hours ago,KYLE DAUKAUS | THE D’ARCE KNIGHT RETURNS,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca...",KYLE DAUKAUS | THE D’ARCE KNIGHT RETURNS,"Kyle Daukaus Opens Up About Anxiety, Discusses...",2025-09-15 15:48:25-05:00,2025-09-15 16:23:45-05:00,E. Spencer Kyte,None,On the morning of the fight he’d been chasing ...,[https://ufc.com/images/styles/background_imag...
2,/news/live-results-official-scorecards-match-r...,Results,2 days ago,Results + Scorecards | Canelo vs Crawford,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca...",Results + Scorecards | Canelo vs Crawford,"Get Live Match Results, Official Scorecards, P...",2025-09-14 01:00:00-05:00,2025-09-14 01:34:15-05:00,Thomas Gerbasi,tgerbasi,"Two of boxing’s most dominant forces, undisput...",[https://ufc.com/images/styles/background_imag...
3,/news/main-card-live-results-fight-recaps-high...,Results,2 days ago,Main Card Results | Noche UFC,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca...",Main Card Results | Noche UFC,"See The Fight Results, Watch Post-Fight Interv...",2025-09-13 20:39:00-05:00,2025-09-13 20:55:21-05:00,E. Spencer Kyte,SpencerKyte,Saturday’s Noche UFC main card at Frost Bank C...,[https://ufc.com/images/styles/background_imag...
4,/news/prelim-live-results-fight-recaps-highlig...,Results,2 days ago,Prelim Results | Noche UFC,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca...",Prelim Results | Noche UFC,"See The Fight Results, Watch Post-Fight Interv...",2025-09-13 16:30:00-05:00,2025-09-13 18:59:45-05:00,E. Spencer Kyte,SpencerKyte,The third annual Noche UFC event took place on...,[https://ufc.com/images/styles/background_imag...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
724,/news/umar-nurmagomedov-is-ready-to-go-into-de...,Athletes,8 months ago,Umar Nurmagomedov Is Ready To Go Into Deep Waters,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca...",Umar Nurmagomedov Is Ready To Go Into Deep Waters,Undefeated Russian Challenger Is Prepared To M...,2025-01-13 12:36:33-06:00,2025-01-17 13:36:55-06:00,Simon Head,simonheadsport,It’s taken just three years and six fights for...,[https://ufc.com/images/styles/background_imag...
725,/news/grant-dawson-very-good-year-ufc-311,Athletes,8 months ago,Grant Dawson's Very Good Year,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca...",Grant Dawson's Very Good Year,Lightweight Looks To Ride 2024's Winning Momen...,2025-01-13 11:54:29-06:00,2025-01-13 09:55:07-06:00,Thomas Gerbasi,thomasgerbasi,"Grant Dawson is back. Well, he never left, so ...",[https://ufc.com/images/styles/background_imag...
726,/news/islam-makhachev-wants-to-eliminate-all-d...,Athletes,8 months ago,Islam Makhachev Wants To Eliminate All Doubt,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca...",Islam Makhachev Wants To Eliminate All Doubt,Undisputed UFC Lightweight Champion Wants To L...,2025-01-13 10:52:00-06:00,2025-01-17 13:45:40-06:00,Simon Head,simonheadsport,"*Update: Due to medical issues, non-weight cut...",[https://ufc.com/images/styles/background_imag...
727,/news/prelim-results-highlights-winner-intervi...,Results,8 months ago,Prelim Results | UFC Fight Night: Dern vs Ribas 2,"\n\n\n\n\n\n<!DOCTYPE html>\n<html lang=""en-ca...",Prelim Results | UFC Fight Night: Dern vs Ribas 2,"See The Fight Results, Watch Post-Fight Interv...",2025-01-11 21:08:00-06:00,2025-01-13 12:27:19-06:00,E. Spencer Kyte,spencerkyte,The 2025 slate got underway with a preliminary...,[https://uf

In [38]:
def get_page_name(page_link: str):
    page_name = page_link.replace("/", "_")

    if page_name.startswith("_"):
        page_name = page_name[1:]

    return page_name

In [39]:
def get_image_name(image_src: str):
    return image_src\
        .split("/")[-1]\
        .split("?")[0]

In [73]:
def download_images(links: Iterable[str], images_srcs: Iterable[Iterable[str]]):
    def download_images_on_page(page_link: str, image_srcs: Iterable[str]):
        folder = f"images/{get_page_name(page_link)}"
        if os.path.exists(folder):
            return
        
        for src in image_srcs:
            filename = get_image_name(src)

            try:
                os.makedirs(folder, exist_ok=True)

                with open(f"{folder}/{filename}", "wb") as file:
                    image_content = get_request(src).content
                    file.write(image_content)
            except Exception as e:
                print("Error downloading image", src)
                print(e)
    
    zipped = zip(links, images_srcs)
    list_len = len(links)

    for (i, (link, image_srcs)) in enumerate(zipped):
        if i % 10 == 0:
            print("Getting images from page", i, "/", list_len)

        download_images_on_page(link, image_srcs)

In [75]:
download_images(articles_df["link"], articles_df["images_srcs"])

Getting images from page 0 / 729
Getting images from page 10 / 729
Getting images from page 20 / 729
Getting images from page 30 / 729
Getting images from page 40 / 729
Getting images from page 50 / 729
Getting images from page 60 / 729
Getting images from page 70 / 729
Getting images from page 80 / 729
Getting images from page 90 / 729
Getting images from page 100 / 729
Getting images from page 110 / 729
Getting images from page 120 / 729
Getting images from page 130 / 729
Getting images from page 140 / 729
Getting images from page 150 / 729
Getting images from page 160 / 729
Getting images from page 170 / 729
Getting images from page 180 / 729
Getting images from page 190 / 729
Getting images from page 200 / 729
Getting images from page 210 / 729
Getting images from page 220 / 729


KeyboardInterrupt: 